# Pricing desk
Pricing is a black box: each method is a Python module in `voltvision/methods/` that declares its own rate card and premium math. This desk resolves those cards, quotes the same simulated book under every regime, and compares — N regimes, no hardcoding.

Books come from `run_scenarios.py` (`shared/results/<scenario>_s<seed>.pkl`); overrides come from scenario `pricing` blobs, never from system code.

In [ ]:
import os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import (load_template, ensure_core_methods, resolve_cards, quote,
                        describe_pricer, register_pricer, summary, lr, lr_by, PRICERS)
from voltvision import io
from voltvision.simulate import simulate_book
ensure_core_methods()  # load the 02x CALC cells
print('registered pricers:', sorted(PRICERS))

## Rate cards (each method module declares its own — nothing to edit here)

In [ ]:
cfg = load_template()
cards = resolve_cards(cfg)
for regime in cfg['regimes']:
    print(f"--- {regime} ({describe_pricer(regime)['label']})")
    display(pd.DataFrame(describe_pricer(regime)['params']))

## Load one result (written by `run_scenarios.py`)

In [ ]:
try:
    SC, SEED_RUN = io.pick_result(('base', 42))   # preferred run, else the newest
    raw = io.sim_book(io.load_result(SC, SEED_RUN))
    print(f'loaded shared/results/{SC}_s{SEED_RUN}: {len(raw)} rows')
except FileNotFoundError:
    print('no results yet - quick inline sim (run `python run_scenarios.py` for real books)')
    quick = {**load_template(), 'n': 1000, 'n_years': 2}
    raw = simulate_book(quick, quick['vehicle_mix'], seed=0, n_years=2)


## Quote — same book, every regime, side by side

In [ ]:
books = {r: quote(raw, r, cards[r], cfg) for r in cfg['regimes']}
display(summary(books))
print('LR by coverage (rows=coverage, cols=regime):')
display(pd.DataFrame({m: lr_by(b, 'COVERAGE_TYPE').round(1) for m, b in books.items()}))
print('LR by vehicle:')
display(pd.DataFrame({m: lr_by(b, 'VEHICLE_TYPE').round(1) for m, b in books.items()}))

## What-if — override cards through a cfg patch (same mechanism scenarios use)

In [ ]:
alt_cfg = {**cfg, 'pricing': {'glm': {'target_lr': 0.50},
                              'telem': {'target_lr': 0.50},
                              'tariff': {'tpo_loading': 1.50}}}
alt_cards = resolve_cards(alt_cfg)
base_lr = {m: round(lr(b), 1) for m, b in books.items()}
alt_books = {m: quote(raw, m, alt_cards[m], alt_cfg) for m in alt_cfg['regimes']}
alt_lr = {m: round(lr(b), 1) for m, b in alt_books.items()}
display(pd.DataFrame({'base LR (%)': base_lr, 'alt LR (%)': alt_lr,
    'delta (pp)': {m: round(alt_lr[m] - base_lr[m], 1) for m in base_lr}}))

## Nth regime — the standard format, copy this pattern for a real method
A method = a module in `voltvision/methods/` exporting `CARD` and a function `(book, card, cfg, base_cfg) -> book + FINAL_PREMIUM_SST`, registering itself on import. The loader picks up every module automatically.

In [ ]:
def price_flat_demo(book, card, cfg, base_cfg=None):
    """DEMO: same premium for every policy."""
    out = book.copy()
    out['FINAL_PREMIUM_SST'] = float(card.flat_premium)
    return out

register_pricer('flat_demo', price_flat_demo,
    card={'flat_premium': {'default': 1500.0, 'unit': 'RM',
                           'note': 'flat premium charged to everyone'}},
    info={'label': 'Flat (demo)'})
regimes_n = [*cfg['regimes'], 'flat_demo']
cards_n = resolve_cards(cfg, regimes=regimes_n)
books_n = {m: quote(raw, m, cards_n[m], cfg) for m in regimes_n}
display(summary(books_n))

## Notes
- TPO tariff underprices (TPO LR >100%) — raise `tpo_sa_pct`/`tpo_loading` in a scenario pricing blob or in `voltvision/methods/tariff.py`.
- GLM/telem train out-of-sample by default (separate earlier window, base template world); `train_book_seed: null` reverts to in-sample.
- All figures, diagnostics and realism checks live in `analysis.ipynb`.